# Biodiversity loss in the Brazilian Amazon

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pysrc.optimization import solve_planner_problem
from pysrc.services.data_service import load_site_data, load_productivity_params

## Load data

In [2]:
# Model hyperparameters
solver = "gurobi"
pee = 6.6
pa = 41.11
num_sites = 1043
T = 200

In [3]:
# Load site data
(zbar_2017, z_2017, forest_area_2017) = load_site_data(num_sites)

# Load baseline productivity params
(theta, gamma) = load_productivity_params(num_sites)

# Load biodiversity per hectare
eta = pd.read_csv("../data/eta_params.csv")

# Compute initial carbon stock
x_2017 = gamma * forest_area_2017

## Solve model under reforestation counterfactual

In [4]:
# Solve planner problem
res_25 = solve_planner_problem(
    x0=x_2017,
    z0=z_2017,
    zbar=zbar_2017,
    gamma=gamma,
    theta=theta,
    time_horizon=T,
    price_cattle=pa,
    price_emissions=pee + 25,
)

Solving the optimization problem...
Set parameter Username
Academic license - for non-commercial use only - expires 2025-11-04
Read LP format model from file /var/folders/d1/k6sr3htd7fdgk614mmq8pwq00000gr/T/tmpdut3mads.pyomo.lp
Reading time = 1.11 seconds
x1: 417600 rows, 834801 columns, 2083271 nonzeros
Gurobi Optimizer version 10.0.3 build v10.0.3rc0 (mac64[arm])

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 417600 rows, 834801 columns and 2083271 nonzeros
Model fingerprint: 0xf6319a80
Model has 400 quadratic objective terms
Coefficient statistics:
  Matrix range     [1e+00, 9e+02]
  Objective range  [9e-04, 6e+03]
  QObjective range [2e+03, 2e+05]
  Bounds range     [3e-06, 1e+00]
  RHS range        [9e-11, 4e-01]
Presolve removed 208947 rows and 209294 columns
Presolve time: 0.35s
Presolved: 208653 rows, 625507 columns, 1249571 nonzeros
Presolved model has 400 quadratic objective terms
Ordering time: 0.02s

B

In [5]:
z_2017.sum() * 1e3

57.262944734251796

In [6]:
(res_25.X[30].sum() - res_25.X[0].sum())

18.33237187543949

## Solve model under deforestation counterfactual

In [7]:
# Solve planner problem
res_0 = solve_planner_problem(
    x0=x_2017,
    z0=z_2017,
    zbar=zbar_2017,
    gamma=gamma,
    theta=theta,
    time_horizon=T,
    price_cattle=pa,
    price_emissions=pee,
)

Solving the optimization problem...
Set parameter Username
Academic license - for non-commercial use only - expires 2025-11-04
Read LP format model from file /var/folders/d1/k6sr3htd7fdgk614mmq8pwq00000gr/T/tmptvygenmu.pyomo.lp
Reading time = 1.07 seconds
x1: 417600 rows, 834801 columns, 2083271 nonzeros
Gurobi Optimizer version 10.0.3 build v10.0.3rc0 (mac64[arm])

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 417600 rows, 834801 columns and 2083271 nonzeros
Model fingerprint: 0x9b469537
Model has 400 quadratic objective terms
Coefficient statistics:
  Matrix range     [1e+00, 9e+02]
  Objective range  [1e-03, 1e+03]
  QObjective range [2e+03, 2e+05]
  Bounds range     [3e-06, 1e+00]
  RHS range        [9e-11, 4e-01]
Presolve removed 208947 rows and 209294 columns
Presolve time: 0.36s
Presolved: 208653 rows, 625507 columns, 1249571 nonzeros
Presolved model has 400 quadratic objective terms
Ordering time: 0.01s

B

In [19]:
from pysrc.analysis import value_decomposition

v_25 = value_decomposition(T=30, pee=6.60, pa=41.11, b=25, theta=theta, solution=res_25)["total_PV"]
v_0 = value_decomposition(T=30, pee=6.60, pa=41.11, b=0, theta=theta, solution=res_0)["total_PV"]
v_25 - v_0

376.17528331065665

In [12]:
v_0

57.208661782145086

In [18]:
value_decomposition(T=200, pee=6.60, pa=41.11, b=0, theta=theta, solution=res_0)

{'pa': 41.11,
 'pee': 6.6,
 'b': 0,
 'total_AO': 364.01842546703995,
 'total_NT': 0.0,
 'total_FS': -113.80582690138974,
 'total_AC': 5.836771550353814,
 'total_PV': 244.3758270152964}

In [15]:
value_decomposition(T=200, pee=6.60, pa=41.11, b=25, theta=theta, solution=res_25)

{'pa': 41.11,
 'pee': 6.6,
 'b': 25,
 'total_AO': 14.848613175159521,
 'total_NT': 422.17015907646174,
 'total_FS': 111.45292199618588,
 'total_AC': 21.86891864504492,
 'total_PV': 526.6027756027622}

## Analysis

In [8]:
def compute_biodiversity_changes(res, T=30):
    # Compute initial biodiversity
    f0 = 1 - (res.Z[0] / zbar_2017)
    fT = 1 - (res.Z[T] / zbar_2017)
    ib = eta * f0

    # Initialize biodiversity law of motion
    B = np.zeros((T, num_sites))

    # Calibrate growth rate
    alpha = 1 - (1 - 0.90) ** (1 / 32)

    # Compute law of motion
    t = np.arange(0, T)
    B = eta[:, np.newaxis] * (1 - np.exp(-alpha * t))

    # Reverse - site reforested in t = 1 has biodiversity of age 29
    B = B[:, ::-1]

    # Get land use changes
    Z_dot = np.diff(res.Z, axis=0)

    # Gains from reforestation
    W = -Z_dot / zbar_2017
    W[W < 0] = 0
    W = W[:T]

    ab_ref = (W.T * B).sum(axis=1).copy()

    # Losses from deforestation
    lb_def = eta * np.maximum(-(fT - f0), 0)

    # New biodiversity
    nb = ib + ab_ref - lb_def

    # Total biodiversity (SAR)
    F0 = zbar_2017 - res.Z[0]
    FT = zbar_2017 - res.Z[T]

    print("mean AT/A0", (FT / F0).mean())
    pS = ((FT / F0).round(4)) ** 0.25

    # Carbon

    return pd.DataFrame(
        {
            "f0": f0.round(4),
            "fT": fT.round(4),
            "new_bio_ha": nb,
            "initial_bio_ha": ib,
            "gain_bio_ha": ab_ref,
            "pct_gain_bio_ha": 100 * (ab_ref / ib),
            "loss_bio_ha": lb_def,
            "pct_loss_bio_ha": 100 * (lb_def / ib),
            "change_bio_ha": ab_ref - lb_def,
            "pct_change_bio_ha": 100 * (ab_ref - lb_def) / ib,
            "pct_change_bio_SAR": 100 * (pS - 1),
            "additional_pct_change_carbon": 100 * ((nb / ib) ** 0.26 - 1) * fT,
        }
    )

## Reforestation counterfactual results


In [ ]:
bio_25 = compute_biodiversity_changes(res_25)

bio_25.describe([0.2, 0.5, 0.8]).round(2)

## Deforestation counterfactual results


In [ ]:
bio_0 = compute_biodiversity_changes(res_0)

bio_0.describe([0.2, 0.5, 0.8]).round(2)

## Save counterfactual results


In [11]:
(
    bio_0.join(bio_25, lsuffix="_b_0", rsuffix="_b_25")
    .describe([0.1, 0.5, 0.9])
    .round(2)[
        [
            "pct_change_bio_ha_b_0",
            "pct_change_bio_ha_b_25",
        ]
    ]
    .T.drop(columns=["count", "std"])
    .to_latex("../results/pct_change_bio_ha.tex", float_format="%.2f")
)

In [12]:
(
    bio_0.join(bio_25, lsuffix="_b_0", rsuffix="_b_25")
    .describe([0.1, 0.5, 0.9])
    .round(2)[
        [
            "pct_change_bio_SAR_b_0",
            "pct_change_bio_SAR_b_25",
        ]
    ]
    .T.drop(columns=["count", "std"])
    .to_latex("../results/pct_change_total_bio.tex", float_format="%.2f")
)

## Historical deforestation results

In [19]:
# Load site data
(_, z_1985, forest_area_1985) = load_site_data(num_sites, year=1985)

f0 = 1 - (z_1985 / zbar_2017)
fT = 1 - (z_2017 / zbar_2017)

ib = eta * f0
lb_def = eta * np.maximum(-(fT - f0), 0)

# Total biodiversity (SAR)
F0 = zbar_2017 - z_1985
FT = zbar_2017 - z_2017

pS = ((FT / F0).round(4)) ** 0.25

bio_hist = pd.DataFrame()

# bio_hist["pct_loss_bio_ha"] = 100 * (-lb_def) / ib
bio_hist["pct_change_bio_SAR"] = 100 * (pS - 1)

# bio_hist["+_pct_change_carbon"] = 100 * (((ib - lb_def) / ib) ** 0.26 - 1) * fT
bio_hist["+_pct_change_carbon_SAR"] = 100 * (pS**0.26 - 1) * fT

X_2017 = (zbar_2017 - z_2017) * gamma
X_1985 = (zbar_2017 - z_1985) * gamma

bio_hist["inital_pct_change_carbon"] = 100 * ((X_2017 - X_1985) / X_1985)

(
    bio_hist.describe([0.1, 0.5, 0.9])
    .round(2)
    .T.drop(columns=["count", "std"])
    .to_latex("../results/biomass_change_hist.tex", float_format="%.1f")
)

In [ ]:
bio_hist["+biomassloss_co2e"] = (bio_hist["+_pct_change_carbon_SAR"] / 100) * X_1985

print(bio_hist["+biomassloss_co2e"].describe())
print(bio_hist["+biomassloss_co2e"].sum())

In [17]:
bio_hist.to_csv("../results/bio_hist.csv")